In [1]:
# from db import get_session_import, engine_import, Base_import,get_session_export, engine_export, Base_export
# from db_exporter import export_table, export_table_to_csv, export_table_to_sql
# from db_importer import import_sql, import_sql_file
import pandas as pd
from transformer import (
    TransformationConfig,
    replace_column_with_lookup,
    remove_columns,
    add_computed_column,
    add_missing_columns,
)
from sql_writer import csv_to_sql, dataframe_to_batched_inserts, write_dataframe_as_batched_inserts,dataframe_to_insert_statements
from insert import execute_sql_file
from datetime import datetime


# Ensure tables exist
# Base_import.metadata.create_all(bind=engine_import)
# Base_export.metadata.create_all(bind=engine_export)
# print("Database tables ready.")

In [25]:
from typing import Any, Callable, Mapping
import random
import string
def can_cast_int(column: str) -> Callable[[Mapping[str, Any]], bool]:
    def _predicate(row: Mapping[str, Any]) -> bool:
        value = row.get(column)
        if value is None or pd.isna(value):
            return False
        try:
            int(value)
            return True
        except (ValueError, TypeError):
            return False
    return _predicate

def cast_int(value):
    try:
        return int(value)
    except (ValueError, TypeError):
        return value  # Return original value if it cannot be cast


def random_8_letters(_row=None):
    letters = string.ascii_letters  # 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'
    return ''.join(random.choices(letters, k=8))

def random_email(_row=None):
    letters = string.ascii_letters  # 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'
    return f"{''.join(random.choices(letters, k=8))}@mail.com"

In [30]:
import pandas as pd

pdinventory =pd.read_csv("snd_inventory_master.csv")
invtypeconfig = TransformationConfig(
    row_filters=[],
    replace_lookups=[],
    remove_columns=[
        'min_order_qty',
       'price_per_unit','discount', 'min_variance',
       'max_variance', 'auto_renewal_flag', 'erp_inventory_id',
       'pickup_location_id', 'umo_value', 'category', 'face_value',
       'min_variance_disc', 'max_variance_disc', 'tax_plan', 'dealer_discount',
       'return_policy', 'due_date', 'ziada_points', 'esd_signature',
       'batch_size', 'prefix', 'serial_counter', 'pallet_size', 'available',
       'w_period', 'prod_family', 'price_group', 'work_category', 'equip_type',
       'used_stock', 'sim_card', 'item_type', 'vendor_id', 'type_id',
       'damage_code', 'service_code', 'v_item_type', 'used_item_code',
       'w_avg_ucost', 'calc_wac', 'quantity', 'add_date', 'prevwavg_ucost',
       'mod_date', 'v_item_code', 'v_item_no', 'available_date', 'expiry_date',
       'consigment_po', 'status', 'DEPOSIT_PRICE', 'ADD_BY', 'MOD_BY',
       'national_material_code', 'wg_material_code', 'SUB_CATEGORY', 'weight',
       'return_flag', 'color', 'size', 'is_bundle', 'parent_id', 'send_to_bi',
       'remarks'
    ],computed_columns=[],
    required_columns={
       
    },
    replace_columns=[
      
    ]
)


transformed_invtyp = invtypeconfig.apply(pdinventory.copy().drop_duplicates(subset=["inventory_type_id"]), None)
transformed_invtyp["inventory_type_id"] = transformed_invtyp["inventory_type_id"].map(cast_int)
# print(transformed_invtyp.head(10))
write_dataframe_as_batched_inserts(
    df=transformed_invtyp,
    table_name="inventory_types",
    output_path="snd/inventory_type.sql",
    batch_size=1000
)   

# pdinventory.head()


In [31]:
pdinventory.head()

,inventory_type_id,inventory_type_description,min_order_qty,price_per_unit,inventory_sub_type,discount,min_variance,max_variance,auto_renewal_flag,erp_inventory_id,...,wg_material_code,SUB_CATEGORY,weight,return_flag,color,size,is_bundle,parent_id,send_to_bi,remarks
0,1.0,Upfront Payment,1.0,10.0,P,0.0,NaN,NaN,N,UPFRONTPAYMENT,...,NaN,Non Voucher Product,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,Deposit Amount,1.0,10.0,P,0.0,NaN,NaN,N,DEPOSIT,...,NaN,Non Voucher Product,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,Ethiopia 128k Pre-paid SIM Starter Pack – EBU DSA,NaN,0.0,P,0.0,NaN,NaN,Y,ETA00004,...,NaN,Non Voucher Product,1.0,Y,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,BAMBA 10 Batch 1000,1.0,10.0,P,0.0,NaN,NaN,Y,S0200011,...,NaN,Voucher Product,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,Voucher card 15 ETB,1.0,10.0,P,0.0,NaN,NaN,Y,VC15,...,NaN,Voucher Product,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
# len(pdinventory["inventory_type_id"].unique())
# pdinventory.columns
# pdinventory.copy().drop_duplicates(subset=["inventory_type_id"]).head()
transformed_invtyp.head()

,inventory_type_id,inventory_type_description,inventory_sub_type
